In [6]:
# =========================
# Standard library imports
# =========================
import sys
import os
import io
import gzip
from pathlib import Path
import math
import json
import pickle
import logging
import random
import subprocess
import warnings
import collections
import itertools
import re
from IPython.display import display

warnings.filterwarnings("ignore")


# =========================
# Numeric / stats
# =========================
import numpy as np
import pandas as pd
from scipy import sparse
from scipy import stats
from scipy.stats import rankdata
from scipy import __version__ as scipy_version


# =========================
# Plotting / visualization
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, FormatStrFormatter, MaxNLocator
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
import seaborn as sns


# =========================
# scikit-learn
# =========================
from sklearn import model_selection, metrics
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    KFold,
    GroupKFold,
    cross_val_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import LocalOutlierFactor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn import __version__ as sklearn_version


# =========================
# Optimization / AutoML
# =========================
import optuna


# =========================
# LightGBM
# =========================
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    _HAS_LGBM = True
except Exception as e:
    _HAS_LGBM = False
    raise RuntimeError(
        "LightGBM not installed. Please install it with `pip install lightgbm`."
    ) from e


# =========================
# RNA structure (ViennaRNA)
# =========================
import RNA


# =========================
# UpSet plots
# =========================
try:
    from upsetplot import UpSet, from_memberships
except Exception:
    _ = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "upsetplot"],
        check=False,
    )
    from upsetplot import UpSet, from_memberships


# =========================
# Model explanation
# =========================
import shap


# =========================
# User-provided utilities
# =========================
import sylib  


# =========================
# Version info
# =========================
print(f"python    = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas    = {pd.__version__}")
print(f"numpy     = {np.__version__}")
print(f"scipy     = {scipy_version}")
print(f"optuna    = {optuna.__version__}")
print(f"sklearn   = {sklearn_version}")
print(f"ViennaRNA = {RNA.__version__}")
print(f"lightgbm  = {lgb.__version__}")
print(f"sylib     = {sylib.__version__}")


# =========================
# Progress bar & logging
# =========================
# progress bar from sylib
progress_bar = sylib.utils.ProgressBar()

# reset handlers then configure logging
logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)
logger = logging.getLogger(__name__)

# make matplotlib quieter
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python    = 3.11.15
pandas    = 2.3.3
numpy     = 2.4.6
scipy     = 1.17.1
optuna    = 4.9.0
sklearn   = 1.9.0
ViennaRNA = 2.7.2
lightgbm  = 4.6.0
sylib     = 0.3.0.dev0+ae18bb2


In [7]:
# ============================================
# Global plotting configuration
# ============================================

_PLOT_CFG = {
    "fig_w": 6.0,
    "fig_h": 6.0,
    "dpi": 300,
}


SPECIES_INFO = {
    "AT21": {
        "label": "AT",
        "short": "AT21",
        "color": "#664D0AFF",
        "marker": "o",
    },
    "NB21": {
        "label": "NB",
        "short": "NB21",
        "color": "#7e3131",
        "marker": "^",
    },
    "OS21": {
        "label": "OS",
        "short": "OS21",
        "color": "#13563f",
        "marker": "s",
    },
}

def set_plot_style(
    *,
    base_fontsize=10,
    title_fontsize=12,
    label_fontsize=10,
    tick_fontsize=9,
    legend_fontsize=10,
    dpi=300,
    axes_linewidth=1.2,
    spines_top=True,
    spines_right=True,
    tick_size_major=6,
    tick_dir="out",
    grid=False,
    fig_w=6.0,
    fig_h=6.0,
):
    sns.set_style("ticks")

    mpl.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": base_fontsize,

        "axes.titlesize": title_fontsize,
        "axes.labelsize": label_fontsize,

        "xtick.labelsize": tick_fontsize,
        "ytick.labelsize": tick_fontsize,

        "legend.fontsize": legend_fontsize,

        "figure.dpi": dpi,
        "savefig.dpi": dpi,

        "axes.linewidth": axes_linewidth,
        "axes.spines.top": spines_top,
        "axes.spines.right": spines_right,
        "axes.grid": grid,
        "axes.axisbelow": True,

        "xtick.major.size": tick_size_major,
        "ytick.major.size": tick_size_major,
        "xtick.direction": tick_dir,
        "ytick.direction": tick_dir,

        "legend.frameon": False,

        "savefig.bbox": "tight",
        "savefig.transparent": False,
        "figure.autolayout": False,
    })

    _PLOT_CFG.update({
        "fig_w": fig_w,
        "fig_h": fig_h,
        "dpi": dpi,
    })


def make_fig(w=None, h=None, dpi=None):
    W = float(w) if w is not None else _PLOT_CFG["fig_w"]
    H = float(h) if h is not None else _PLOT_CFG["fig_h"]
    D = dpi if dpi is not None else _PLOT_CFG["dpi"]

    fig, ax = plt.subplots(
        figsize=(W, H),
        dpi=D,
    )

    return fig, ax


def _compact_formatter():
    def _fmt(x, _pos=None):
        axx = abs(x)

        if axx >= 1e9:
            s = f"{x / 1e9:.1f}B"
        elif axx >= 1e6:
            s = f"{x / 1e6:.1f}M"
        elif axx >= 1e3:
            s = f"{x / 1e3:.1f}k"
        else:
            s = f"{x:.2g}"

        return (
            s.replace(".0B", "B")
             .replace(".0M", "M")
             .replace(".0k", "k")
        )

    return FuncFormatter(_fmt)


def format_axis(
    ax,
    *,
    xlabel=None,
    ylabel=None,
    compact_ticks=(),
):
    if xlabel is not None:
        ax.set_xlabel(xlabel)

    if ylabel is not None:
        ax.set_ylabel(ylabel)

    fmt = _compact_formatter()

    if "x" in compact_ticks:
        ax.xaxis.set_major_formatter(fmt)

    if "y" in compact_ticks:
        ax.yaxis.set_major_formatter(fmt)

    return ax


# ============================================
# Joint scatter with KDE marginals
# ============================================

def safe_pearsonr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.pearsonr(x, y)


def safe_spearmanr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.spearmanr(x, y)

def _kde_1d(values, lo, hi, num=256):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    grid = np.linspace(lo, hi, num)

    if len(values) < 2:
        return grid, np.zeros_like(grid)

    try:
        kde = stats.gaussian_kde(values)
        dens = kde(grid)
        dens /= dens.max() if dens.max() > 0 else 1
        return grid, dens

    except Exception:
        return grid, np.zeros_like(grid)


def joint_scatter(
    x,
    y,
    *,
    color=None,
    point_size=18,
    alpha=0.65,
    show_identity=True,
    show_regression=True,
    annotate=True,
    annotate_spearman=True,
    xlabel=None,
    ylabel=None,
    title=None,
    figsize=None,
    w=None,
    h=None,
    dpi=None,
    annotate_fontsize=14,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)

    if figsize is not None:
        FW, FH = figsize
    else:
        FW = float(w) if w is not None else _PLOT_CFG["fig_w"]
        FH = float(h) if h is not None else _PLOT_CFG["fig_h"]

    fig = plt.figure(
        figsize=(FW, FH),
        dpi=(dpi or _PLOT_CFG["dpi"]),
    )

    gs = GridSpec(
        2,
        2,
        width_ratios=(4, 1),
        height_ratios=(1, 4),
        hspace=0.05,
        wspace=0.05,
    )

    ax_top = fig.add_subplot(gs[0, 0])
    ax_joint = fig.add_subplot(gs[1, 0], sharex=ax_top)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)

    ax_joint.scatter(
        x,
        y,
        s=point_size,
        alpha=alpha,
        edgecolor="none",
        color=color,
    )

    lo = float(np.nanmin([x.min(), y.min()]))
    hi = float(np.nanmax([x.max(), y.max()]))

    pad = 0.05 * (hi - lo if hi > lo else 1.0)

    lo -= pad
    hi += pad

    ax_joint.set_xlim(lo, hi)
    ax_joint.set_ylim(lo, hi)

    if show_identity:
        ax_joint.plot(
            [lo, hi],
            [lo, hi],
            ls="--",
            lw=1.2,
            color="0.65",
            zorder=1,
        )

    if show_regression and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        slope, intercept = np.polyfit(x, y, 1)

        ax_joint.plot(
            [lo, hi],
            slope * np.array([lo, hi]) + intercept,
            color="black",
            lw=1.5,
            zorder=2,
        )

    format_axis(
        ax_joint,
        xlabel=xlabel,
        ylabel=ylabel,
        compact_ticks=(),
    )

    if title:
        ax_joint.set_title(title)

    if annotate and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        rp, _ = safe_pearsonr(x, y)
        rs, _ = safe_spearmanr(x, y)

        txt = rf"$r_p = {rp:.2f}$"

        if annotate_spearman:
            txt += "\n" + rf"$r_s = {rs:.2f}$"

        txt += f"\n$n = {n}$"

        ax_joint.text(
            0.04,
            0.96,
            txt,
            transform=ax_joint.transAxes,
            ha="left",
            va="top",
            fontsize=annotate_fontsize,
        )

    gx, dx = _kde_1d(x, lo, hi)
    gy, dy = _kde_1d(y, lo, hi)

    ax_top.plot(gx, dx, lw=2, color=color)
    ax_top.axis("off")

    ax_right.plot(dy, gy, lw=2, color=color)
    ax_right.axis("off")

    plt.tight_layout()

    return fig, (ax_joint, ax_top, ax_right)


set_plot_style()

# ============================================
# Plot helper functions
# ============================================

def species_palette():
    return [
        SPECIES_INFO["AT21"]["color"],
        SPECIES_INFO["NB21"]["color"],
        SPECIES_INFO["OS21"]["color"],
    ]


def outside_legend(
    ax,
    *,
    title="Species",
    n_items=3,
    x=1.02,
    y=1.00,
):
    handles, labels = ax.get_legend_handles_labels()

    ax.legend(
        handles[:n_items],
        labels[:n_items],
        title=title,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(x, y),
        borderaxespad=0,
    )

    return ax

In [8]:
# ============================================
# Paths
# ============================================

RESULT_DIR = Path(
    "/mnt/d/Ibnu/Programming/data/regression/results/lgbm"
)

SPECIES = [
    "AT21",
    "NB21",
    "OS21",
]

# --------------------------------------------
# Candidate table
# --------------------------------------------

candidate_file = (
    RESULT_DIR
    / (
        "lgbm.species_specific_candidate_table"
        ".importance_p90.sss_p90.tsv.gz"
    )
)

candidate_all_df = pd.read_csv(
    candidate_file,
    sep="\t",
)

candidate_selected_df = (
    candidate_all_df.loc[
        candidate_all_df["Selected"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Candidate table")
print("----------------")
print("All rows :", len(candidate_all_df))
print("Selected :", len(candidate_selected_df))

display(
    candidate_selected_df.head()
)


# --------------------------------------------
# Load species-specific LightGBM packages
# --------------------------------------------

lgbm_models = {}
lgbm_data = {}

for species in SPECIES:

    model_file = (
        RESULT_DIR
        / f"{species}.lgbm.model.txt"
    )

    bundle_file = (
        RESULT_DIR
        / f"{species}.lgbm.data_bundle.pkl.gz"
    )

    # fitted LightGBM model
    model = lgb.Booster(
        model_file=str(model_file)
    )

    # reusable model data
    with gzip.open(
        bundle_file,
        "rb",
    ) as f:
        bundle = pickle.load(f)

    # ----------------------------------------
    # Integrity checks
    # ----------------------------------------

    assert bundle["species"] == species

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_train_model"].columns)
    )

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_test_model"].columns)
    )

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_train_raw"].columns)
    )

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_test_raw"].columns)
    )

    lgbm_models[species] = model
    lgbm_data[species] = bundle

    print()
    print("=" * 60)
    print(species)
    print("Train :", bundle["x_train_model"].shape)
    print("Test  :", bundle["x_test_model"].shape)
    print("Features:", len(bundle["feature_names"]))


print()
print("All reusable LightGBM packages loaded successfully.")

Candidate table
----------------
All rows : 1347
Selected : 70


,Species,Feature,Region,Feature_type,Importance_raw,Importance_normalized,Importance_percentile,SSS,SSS_percentile,Selected
0,AT,5'UTR.Length,5'UTR,Length,21.0,0.005526,0.974388,0.010997,0.997773,True
1,AT,mRNA.MFE,mRNA,RNA_structure_MFE,19.0,0.005000,0.958797,0.008363,0.993318,True
2,AT,3'UTR.UAG-freq,3'UTR,Nucleotide_kmer_freq,21.0,0.005526,0.974388,0.006610,0.986637,True
3,AT,5'UTR.U-freq,5'UTR,Nucleotide_kmer_freq,26.0,0.006842,0.993318,0.006238,0.979955,True
4,AT,CDS.UAU-freq,CDS,Nucleotide_kmer_freq,17.0,0.004474,0.927617,0.005743,0.973274,True



AT21
Train : (5698, 464)
Test  : (1428, 464)
Features: 464

NB21
Train : (4305, 464)
Test  : (1076, 464)
Features: 464

OS21
Train : (4445, 463)
Test  : (1107, 463)
Features: 463

All reusable LightGBM packages loaded successfully.


In [9]:
# ============================================
# Check candidate features against
# the three LightGBM feature spaces
# ============================================

# --------------------------------------------
# Feature sets for each fitted model
# --------------------------------------------

model_feature_sets = {
    species: set(
        lgbm_data[species]["feature_names"]
    )
    for species in SPECIES
}


# --------------------------------------------
# Selected candidate features
# --------------------------------------------

selected_features = set(
    candidate_selected_df["Feature"]
)

print(
    "Selected candidate rows   :",
    len(candidate_selected_df),
)

print(
    "Unique selected features  :",
    len(selected_features),
)


# --------------------------------------------
# Check selected features in each model
# --------------------------------------------

rows = []

for feature in sorted(selected_features):

    rows.append({
        "Feature": feature,
        "AT21": feature in model_feature_sets["AT21"],
        "NB21": feature in model_feature_sets["NB21"],
        "OS21": feature in model_feature_sets["OS21"],
    })

candidate_feature_check_df = pd.DataFrame(rows)

candidate_feature_check_df["Present_all_species"] = (
    candidate_feature_check_df[
        [
            "AT21",
            "NB21",
            "OS21",
        ]
    ]
    .all(axis=1)
)

display(
    candidate_feature_check_df.head()
)


# --------------------------------------------
# Summary
# --------------------------------------------

print()
print("Candidate feature availability")
print("------------------------------")

print(
    "Present in AT21:",
    candidate_feature_check_df["AT21"].sum(),
)

print(
    "Present in NB21:",
    candidate_feature_check_df["NB21"].sum(),
)

print(
    "Present in OS21:",
    candidate_feature_check_df["OS21"].sum(),
)

print(
    "Present in all three:",
    candidate_feature_check_df[
        "Present_all_species"
    ].sum(),
)


# --------------------------------------------
# Show any missing candidate features
# --------------------------------------------

missing_candidate_df = (
    candidate_feature_check_df.loc[
        ~candidate_feature_check_df[
            "Present_all_species"
        ]
    ]
    .copy()
)

print()
print(
    "Candidate features missing from"
    " at least one model:",
    len(missing_candidate_df),
)

display(
    missing_candidate_df
)

Selected candidate rows   : 70
Unique selected features  : 61


,Feature,AT21,NB21,OS21,Present_all_species
0,3'UTR.ACA-freq,True,True,True,True
1,3'UTR.AG-freq,True,True,True,True
2,3'UTR.AUC-freq,True,True,True,True
3,3'UTR.G-freq,True,True,True,True
4,3'UTR.GAU-freq,True,True,True,True



Candidate feature availability
------------------------------
Present in AT21: 61
Present in NB21: 61
Present in OS21: 61
Present in all three: 61

Candidate features missing from at least one model: 0


,Feature,AT21,NB21,OS21,Present_all_species


In [12]:
# ============================================
# Build raw candidate-feature dataset
# using saved LightGBM data bundles
# ============================================

candidate_features = sorted(candidate_selected_df["Feature"].unique())

print(f"Unique candidate features: {len(candidate_features)}")

raw_candidate_wide = {}

for species in SPECIES:

    bundle = lgbm_data[species]

    # ----------------------------------------
    # Combine raw feature values
    # ----------------------------------------

    x_train_raw = bundle["x_train_raw"].loc[:, candidate_features].copy()
    x_test_raw = bundle["x_test_raw"].loc[:, candidate_features].copy()

    x_raw = pd.concat([x_train_raw, x_test_raw], axis=0)

    # ----------------------------------------
    # Combine metadata
    # ----------------------------------------

    train_meta = bundle["train_metadata"].copy()
    test_meta = bundle["test_metadata"].copy()

    metadata = pd.concat([train_meta, test_meta], axis=0)

    # ----------------------------------------
    # Integrity check
    # ----------------------------------------

    assert x_raw.index.equals(metadata.index)

    # ----------------------------------------
    # Build species dataframe
    # ----------------------------------------

    raw_df = pd.concat([metadata, x_raw], axis=1)

    raw_df.insert(0, "Species", SPECIES_INFO[species]["label"])
    raw_df.insert(1, "Species_key", species)

    raw_candidate_wide[species] = raw_df

# ============================================
# Combine all species
# ============================================

candidate_raw_wide_df = pd.concat(
    raw_candidate_wide.values(),
    ignore_index=True,
)

print("\nCombined raw candidate-feature dataset")
print("--------------------------------------")
print(f"Shape: {candidate_raw_wide_df.shape}")

print("\nSpecies counts")
display(candidate_raw_wide_df["Species"].value_counts())
print(candidate_raw_wide_df.head())
# ============================================
# Descriptive summary of raw candidate features
# by species
# ============================================

candidate_feature_summary_df = (
    candidate_raw_wide_df
    .groupby("Species")[candidate_features]
    .agg(["count", "mean", "std", "median", "min", "max"])
)

display(candidate_feature_summary_df)

Unique candidate features: 61

Combined raw candidate-feature dataset
--------------------------------------
Shape: (18059, 70)

Species counts


Species
AT    7126
OS    5552
NB    5381
Name: count, dtype: int64

  Species Species_key                         var_id     trans_id    gene_id  \
0      AT        AT21    AT5G05370.1.1591901.1590815  AT5G05370.1  AT5G05370   
1      AT        AT21    AT5G16060.1.5246121.5247413  AT5G16060.1  AT5G16060   
2      AT        AT21  AT2G34160.1.14426246.14427367  AT2G34160.1  AT2G34160   
3      AT        AT21  AT5G54600.1.22183004.22184509  AT5G54600.1  AT5G54600   
4      AT        AT21    AT2G23340.1.9937988.9938873  AT2G23340.1  AT2G23340   

    dataset  observed_raw  observed_model  predicted_model  3'UTR.ACA-freq  \
0  Training      0.632897        0.054841         0.418203        6.172840   
1  Training      0.365525       -1.094430        -0.499050        9.803922   
2  Training      0.600538       -0.073162        -0.132563       13.605442   
3  Training      0.544256       -0.302488        -0.542934        0.000000   
4  Training      1.047809        1.491571         0.360980       19.108280   

   ...  mRNA.GCU-freq  mRNA.GU-freq  mRNA.GUA-freq

3'UTR.ACA-freq                                                  \
                 count       mean       std    median  min         max   
Species                                                                  
AT                7126  11.284752  9.932164  9.708738  0.0   68.965517   
NB                5381  10.003368  8.471785  8.620690  0.0  104.166667   
OS                5552  11.296883  8.823811  9.803922  0.0   52.884615   

        3'UTR.AG-freq                                   ... mRNA.UC-freq  \
                count       mean        std     median  ...          std   
Species                                                 ...                
AT               7126  43.455944  18.092570  42.372881  ...    14.261474   
NB               5381  47.933791  17.074809  47.904192  ...    11.443883   
OS               5552  49.043126  16.686696  48.913043  ...    13.257842   

                                          mRNA.Y-freq                         \
            median        min         max       count        mean        std   
Species                                                                        
AT       72.139303  26.178010  155.102041        7126  493.954042  36.211605   
NB       59.625213  27.164686  110.619469        5381  489.981309  34.913898   
OS       68.435754  18.261965  132.231405        5552  502.223150  32.356568   

                                             
             median         min         max  
Species                                      
AT       493.281067  332.425068  672.881356  
NB       489.169675  362.318841  627.952756  
OS       503.089295  377.740304  624.772313  

[3 rows x 366 columns]

In [13]:
# ============================================
# Long-format candidate feature summary
# ============================================

summary_rows = []

for species in ["AT", "NB", "OS"]:

    df = candidate_raw_wide_df[
        candidate_raw_wide_df["Species"] == species
    ]

    for feature in candidate_features:

        values = df[feature].dropna()

        summary_rows.append({
            "Species": species,
            "Feature": feature,
            "n": len(values),
            "mean": values.mean(),
            "std": values.std(),
            "median": values.median(),
            "q25": values.quantile(0.25),
            "q75": values.quantile(0.75),
            "min": values.min(),
            "max": values.max(),
        })

candidate_feature_summary_long_df = pd.DataFrame(summary_rows)

candidate_feature_summary_long_df = (
    candidate_feature_summary_long_df
    .merge(
        candidate_selected_df[
            ["Feature", "Region", "Feature_type"]
        ]
        .drop_duplicates(),
        on="Feature",
        how="left",
    )
)

display(candidate_feature_summary_long_df.head())

,Species,Feature,n,mean,std,median,q25,q75,min,max,Region,Feature_type
0,AT,3'UTR.ACA-freq,7126,11.284752,9.932164,9.708738,4.830918,16.393443,0.00000,68.965517,3'UTR,Nucleotide_kmer_freq
1,AT,3'UTR.AG-freq,7126,43.455944,18.092570,42.372881,31.421845,54.263566,0.00000,148.148148,3'UTR,Nucleotide_kmer_freq
2,AT,3'UTR.AUC-freq,7126,18.525699,11.346800,17.094017,10.309278,25.196871,0.00000,105.263158,3'UTR,Nucleotide_kmer_freq
3,AT,3'UTR.G-freq,7126,172.249670,34.196736,172.774869,150.326797,193.798450,21.73913,363.636364,3'UTR,Nucleotide_kmer_freq
4,AT,3'UTR.GAU-freq,7126,19.107572,11.747014,18.072289,10.989011,25.613748,0.00000,128.205128,3'UTR,Nucleotide_kmer_freq


In [27]:
# ============================================
# Comprehensive raw candidate-feature summary
# by species
# ============================================

summary_rows = []

for species in ["AT", "NB", "OS"]:

    df = candidate_raw_wide_df[candidate_raw_wide_df["Species"] == species]

    for feature in candidate_features:

        values = pd.to_numeric(df[feature], errors="coerce")
        valid = values.dropna()

        summary_rows.append({
            "Species": species,
            "Feature": feature,
            "n": len(valid),
            "missing_n": values.isna().sum(),
            "missing_pct": values.isna().mean(),
            "zero_n": (valid == 0).sum(),
            "zero_pct": (valid == 0).mean(),
            "mean": valid.mean(),
            "std": valid.std(),
            "median": valid.median(),
            "q25": valid.quantile(0.25),
            "q75": valid.quantile(0.75),
            "IQR": valid.quantile(0.75) - valid.quantile(0.25),
            "min": valid.min(),
            "max": valid.max(),
            "skew": valid.skew(),
        })

candidate_feature_summary_long_df = pd.DataFrame(summary_rows)

feature_metadata_df = (
    candidate_selected_df[
        ["Feature", "Region", "Feature_type"]
    ]
    .drop_duplicates()
)

candidate_feature_summary_long_df = (
    candidate_feature_summary_long_df
    .merge(
        feature_metadata_df,
        on="Feature",
        how="left",
        validate="many_to_one",
    )
)

candidate_feature_summary_long_df = candidate_feature_summary_long_df[
    [
        "Species",
        "Feature",
        "Region",
        "Feature_type",
        "n",
        "missing_n",
        "missing_pct",
        "zero_n",
        "zero_pct",
        "mean",
        "std",
        "median",
        "q25",
        "q75",
        "IQR",
        "min",
        "max",
        "skew",
    ]
]

print("Candidate feature summaries:", candidate_feature_summary_long_df.shape)

display(candidate_feature_summary_long_df.head(20))

print("\nMissing-value summary")
display(
    candidate_feature_summary_long_df.groupby("Species")[
        ["missing_pct", "zero_pct"]
    ].describe()
)

print("\nMost zero-inflated features")
display(
    candidate_feature_summary_long_df
    .sort_values("zero_pct", ascending=False)
    .groupby("Species")
    .head(10)[
        ["Species", "Feature", "Region", "Feature_type", "zero_pct"]
    ]
)

print("\nMost skewed features")
display(
    candidate_feature_summary_long_df
    .assign(abs_skew=lambda x: x["skew"].abs())
    .sort_values("abs_skew", ascending=False)
    .groupby("Species")
    .head(10)[
        ["Species", "Feature", "Region", "Feature_type", "skew"]
    ]
)

Candidate feature summaries: (183, 18)


,Species,Feature,Region,Feature_type,n,missing_n,missing_pct,zero_n,zero_pct,mean,std,median,q25,q75,IQR,min,max,skew
0,AT,3'UTR.ACA-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,1486,0.208532,11.284752,9.932164,9.708738,4.830918,16.393443,11.562525,0.00000,68.965517,1.298749
1,AT,3'UTR.AG-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,40,0.005613,43.455944,18.092570,42.372881,31.421845,54.263566,22.841721,0.00000,148.148148,0.533874
2,AT,3'UTR.AUC-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,417,0.058518,18.525699,11.346800,17.094017,10.309278,25.196871,14.887593,0.00000,105.263158,0.841857
3,AT,3'UTR.G-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,0,0.000000,172.249670,34.196736,172.774869,150.326797,193.798450,43.471652,21.73913,363.636364,-0.021333
4,AT,3'UTR.GAU-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,448,0.062868,19.107572,11.747014,18.072289,10.989011,25.613748,14.624737,0.00000,128.205128,1.073647
5,AT,3'UTR.GUA-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,982,0.137805,14.127441,10.158698,12.738854,6.666667,20.134228,13.467562,0.00000,57.971014,0.740521
6,AT,3'UTR.Length,3'UTR,Length,7126,0,0.0,0,0.000000,166.093040,49.883636,162.000000,133.000000,196.000000,63.000000,22.00000,414.000000,0.620488
7,AT,3'UTR.UAG-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,1655,0.232248,9.115598,7.581238,7.812500,4.255319,13.605442,9.350123,0.00000,59.701493,0.905638
8,AT,3'UTR.UC-freq,3'UTR,Nucleotide_kmer_freq,7126,0,0.0,1,0.000140,71.026575,23.745300,69.230769,55.214724,85.470085,30.255362,0.00000,204.081633,0.544159
9,AT,5'UTR.ACA-freq,5'UTR,Nucleotide_kmer_freq,7126,0,0.0,3176,0.445692,20.594851,28.957420,12.500000,0.000000,31.250000,31.250000,0.00000,300.000000,2.503099



Missing-value summary


missing_pct                                    zero_pct            \
              count mean  std  min  25%  50%  75%  max    count      mean   
Species                                                                     
AT             61.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     61.0  0.160203   
NB             61.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     61.0  0.149631   
OS             61.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     61.0  0.134605   

                                                           
              std  min  25%       50%       75%       max  
Species                                                    
AT       0.232347  0.0  0.0  0.024979  0.280101  0.801010  
NB       0.226062  0.0  0.0  0.022115  0.236945  0.808214  
OS       0.209985  0.0  0.0  0.017291  0.297010  0.843120


Most zero-inflated features


,Species,Feature,Region,Feature_type,zero_pct
148,OS,5'UTR.UAU-freq,5'UTR,Nucleotide_kmer_freq,0.843120
93,NB,5'UTR.UGG-freq,5'UTR,Nucleotide_kmer_freq,0.808214
32,AT,5'UTR.UGG-freq,5'UTR,Nucleotide_kmer_freq,0.801010
14,AT,5'UTR.CCC-freq,5'UTR,Nucleotide_kmer_freq,0.785574
154,OS,5'UTR.UGG-freq,5'UTR,Nucleotide_kmer_freq,0.757385
76,NB,5'UTR.CCG-freq,5'UTR,Nucleotide_kmer_freq,0.714923
31,AT,5'UTR.UGC-freq,5'UTR,Nucleotide_kmer_freq,0.713163
75,NB,5'UTR.CCC-freq,5'UTR,Nucleotide_kmer_freq,0.686490
26,AT,5'UTR.UAU-freq,5'UTR,Nucleotide_kmer_freq,0.681869
15,AT,5'UTR.CCG-freq,5'UTR,Nucleotide_kmer_freq,0.669099



Most skewed features


,Species,Feature,Region,Feature_type,skew
148,OS,5'UTR.UAU-freq,5'UTR,Nucleotide_kmer_freq,4.898903
75,NB,5'UTR.CCC-freq,5'UTR,Nucleotide_kmer_freq,4.426347
26,AT,5'UTR.UAU-freq,5'UTR,Nucleotide_kmer_freq,4.268102
32,AT,5'UTR.UGG-freq,5'UTR,Nucleotide_kmer_freq,4.258789
131,OS,5'UTR.ACA-freq,5'UTR,Nucleotide_kmer_freq,3.549424
31,AT,5'UTR.UGC-freq,5'UTR,Nucleotide_kmer_freq,3.443921
154,OS,5'UTR.UGG-freq,5'UTR,Nucleotide_kmer_freq,3.344191
93,NB,5'UTR.UGG-freq,5'UTR,Nucleotide_kmer_freq,3.245552
11,AT,5'UTR.AGG-freq,5'UTR,Nucleotide_kmer_freq,3.235226
14,AT,5'UTR.CCC-freq,5'UTR,Nucleotide_kmer_freq,3.200090


In [28]:
# ============================================
# Candidate-feature correlation analysis
# Raw biological space + transformed model space
# ============================================

from statsmodels.stats.multitest import multipletests

corr_rows = []
candidate_corr_raw = {}
candidate_corr_model = {}

for species_key in SPECIES:

    species = SPECIES_INFO[species_key]["label"]
    bundle = lgbm_data[species_key]

    # ----------------------------------------
    # Raw biological feature values
    # ----------------------------------------

    raw_df = candidate_raw_wide_df[
        candidate_raw_wide_df["Species"] == species
    ][candidate_features].copy()

    raw_corr = raw_df.corr(method="spearman")
    candidate_corr_raw[species] = raw_corr

    # ----------------------------------------
    # Exact transformed values used by LGBM
    # ----------------------------------------

    model_df = pd.concat(
        [
            bundle["x_train_model"][candidate_features],
            bundle["x_test_model"][candidate_features],
        ],
        axis=0,
    )

    model_corr = model_df.corr(method="pearson")
    candidate_corr_model[species] = model_corr

    # ----------------------------------------
    # Pairwise statistics
    # ----------------------------------------

    species_rows = []

    for i, feature_1 in enumerate(candidate_features):

        for j in range(i + 1, len(candidate_features)):

            feature_2 = candidate_features[j]

            x_raw = raw_df[feature_1].to_numpy(dtype=float)
            y_raw = raw_df[feature_2].to_numpy(dtype=float)

            x_model = model_df[feature_1].to_numpy(dtype=float)
            y_model = model_df[feature_2].to_numpy(dtype=float)

            rho, p_spearman = stats.spearmanr(x_raw, y_raw)
            r_model, p_model = stats.pearsonr(x_model, y_model)

            species_rows.append({
                "Species": species,
                "Feature_1": feature_1,
                "Feature_2": feature_2,
                "Spearman_r": rho,
                "Spearman_p": p_spearman,
                "Model_Pearson_r": r_model,
                "Model_Pearson_p": p_model,
            })

    species_pair_df = pd.DataFrame(species_rows)

    # ----------------------------------------
    # Multiple-testing correction
    # 1,830 tests within each species
    # ----------------------------------------

    species_pair_df["Spearman_q"] = multipletests(
        species_pair_df["Spearman_p"],
        method="fdr_bh",
    )[1]

    species_pair_df["Model_Pearson_q"] = multipletests(
        species_pair_df["Model_Pearson_p"],
        method="fdr_bh",
    )[1]

    corr_rows.append(species_pair_df)

corr_pair_df = pd.concat(
    corr_rows,
    ignore_index=True,
)

# --------------------------------------------
# Add biological annotations
# --------------------------------------------

feature_metadata_df = (
    candidate_selected_df[
        ["Feature", "Region", "Feature_type"]
    ]
    .drop_duplicates()
)

meta_1 = feature_metadata_df.rename(columns={
    "Feature": "Feature_1",
    "Region": "Region_1",
    "Feature_type": "Feature_type_1",
})

meta_2 = feature_metadata_df.rename(columns={
    "Feature": "Feature_2",
    "Region": "Region_2",
    "Feature_type": "Feature_type_2",
})

corr_pair_df = (
    corr_pair_df
    .merge(meta_1, on="Feature_1", how="left", validate="many_to_one")
    .merge(meta_2, on="Feature_2", how="left", validate="many_to_one")
)

corr_pair_df["Same_region"] = corr_pair_df["Region_1"] == corr_pair_df["Region_2"]
corr_pair_df["Same_feature_type"] = corr_pair_df["Feature_type_1"] == corr_pair_df["Feature_type_2"]

# --------------------------------------------
# Overall correlation summary
# --------------------------------------------

corr_summary_df = (
    corr_pair_df
    .groupby("Species")
    .agg(
        n_pairs=("Spearman_r", "size"),
        mean_abs_spearman=("Spearman_r", lambda x: np.mean(np.abs(x))),
        median_abs_spearman=("Spearman_r", lambda x: np.median(np.abs(x))),
        max_abs_spearman=("Spearman_r", lambda x: np.max(np.abs(x))),
        n_abs_r_ge_05=("Spearman_r", lambda x: np.sum(np.abs(x) >= 0.50)),
        n_abs_r_ge_07=("Spearman_r", lambda x: np.sum(np.abs(x) >= 0.70)),
        n_abs_r_ge_09=("Spearman_r", lambda x: np.sum(np.abs(x) >= 0.90)),
        n_FDR_005=("Spearman_q", lambda x: np.sum(x < 0.05)),
        mean_abs_model_r=("Model_Pearson_r", lambda x: np.mean(np.abs(x))),
    )
    .reset_index()
)

print("Correlation summary")
display(corr_summary_df)

# --------------------------------------------
# Strong empirical associations
# --------------------------------------------

strong_corr_df = (
    corr_pair_df[
        (corr_pair_df["Spearman_r"].abs() >= 0.70)
        &
        (corr_pair_df["Spearman_q"] < 0.05)
    ]
    .sort_values(
        ["Species", "Spearman_r"],
        key=lambda x: x.abs() if x.name == "Spearman_r" else x,
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print("\nStrong raw-value correlations: |Spearman r| >= 0.70 and FDR < 0.05")
display(
    strong_corr_df[
        [
            "Species",
            "Feature_1",
            "Feature_2",
            "Spearman_r",
            "Spearman_q",
            "Model_Pearson_r",
            "Region_1",
            "Region_2",
            "Feature_type_1",
            "Feature_type_2",
            "Same_region",
            "Same_feature_type",
        ]
    ]
)

print("\nNumber of strong pairs:", len(strong_corr_df))

Correlation summary


,Species,n_pairs,mean_abs_spearman,median_abs_spearman,max_abs_spearman,n_abs_r_ge_05,n_abs_r_ge_07,n_abs_r_ge_09,n_FDR_005,mean_abs_model_r
0,AT,1830,0.105676,0.062853,1.0,37,13,6,1425,0.109108
1,NB,1830,0.104230,0.066672,1.0,43,13,2,1368,0.107252
2,OS,1830,0.127821,0.084094,1.0,70,20,4,1488,0.127074



Strong raw-value correlations: |Spearman r| >= 0.70 and FDR < 0.05


,Species,Feature_1,Feature_2,Spearman_r,Spearman_q,Model_Pearson_r,Region_1,Region_2,Feature_type_1,Feature_type_2,Same_region,Same_feature_type
0,AT,CDS.Length,CDS.aa_pct_X,-1.000000,0.0,-0.994750,CDS,CDS,Length,Amino_acid_composition,True,False
1,AT,mRNA.Length,mRNA.MFE,-0.968306,0.0,-0.969913,mRNA,mRNA,Length,RNA_structure_MFE,True,False
2,AT,CDS.Length,mRNA.Length,0.923965,0.0,0.925665,CDS,mRNA,Length,Length,False,True
3,AT,CDS.aa_pct_X,mRNA.Length,-0.923965,0.0,-0.919476,CDS,mRNA,Amino_acid_composition,Length,False,False
4,AT,CDS.Length,mRNA.MFE,-0.909533,0.0,-0.913362,CDS,mRNA,Length,RNA_structure_MFE,False,False
5,AT,CDS.aa_pct_X,mRNA.MFE,0.909533,0.0,0.909181,CDS,mRNA,Amino_acid_composition,RNA_structure_MFE,False,False
6,AT,5'UTR.Length,5'UTR.MFE,-0.818495,0.0,-0.828878,5'UTR,5'UTR,Length,RNA_structure_MFE,True,False
7,AT,mRNA.C-freq,mRNA.CC-freq,0.809436,0.0,0.809310,mRNA,mRNA,Nucleotide_kmer_freq,Nucleotide_kmer_freq,True,True
8,AT,5'UTR.CU-freq,5'UTR.CUC-freq,0.793638,0.0,0.769521,5'UTR,5'UTR,Nucleotide_kmer_freq,Nucleotide_kmer_freq,True,True
9,AT,5'UTR.CU-freq,5'UTR.UC-freq,0.757204,0.0,0.738844,5'UTR,5'UTR,Nucleotide_kmer_freq,Nucleotide_kmer_freq,True,True



Number of strong pairs: 46
